<a href="https://colab.research.google.com/github/sulthanalihsan/data-science-2026/blob/main/Pertemuan12_Muhamad_Sulthan_Al_Ihsan_250401020154.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import warnings
# Sembunyikan DeprecationWarning agar output lebih bersih
warnings.filterwarnings('ignore', category=DeprecationWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

# PERTEMUAN 12 : Asosiasi Data & Sistem Rekomendasi Dasar
**Nama  :** Muhammad Sulthan Al Ihsan

**NIM   :** 250401020154

**Mata Kuliah:** Data Science —  S1 PJJ Informatika

**Kelas:** IF401

# Langkah 1: Generate & Eksplorasi Dataset Transaksi

In [17]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
  n_item = np.random.randint(2, 6)
  transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
  if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
    transaksi[i].append('Selai')
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


# Langkah 2: One-Hot Encoding Transaksi
Ubah daftar transaksi menjadi tabel one-hot encoding menggunakan TransactionEncoder.

In [18]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


# Langkah 3: Cari Frequent Itemset dengan Apriori
Jalankan Apriori dengan beberapa nilai min_support, amati bagaimana jumlah itemset yang
ditemukan berubah.

In [19]:
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
  freq = apriori(df, min_support=ms, use_colnames=True)
print(f'min_support={ms}: {len(freq)} itemset ditemukan')
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


Catatan : Cell kode ini bertujuan mencari kombinasi produk yg paling sering muncul dlm transaksi. Pertama, dilakukan uji coba dgn bbrp nilai min_support (ambang batas minimum) untk melihat bagaimana parameter ini memengaruhi jml itemset yg terdeteksi. Langkah ini penting utk menemukan keseimbangan hasil agar tdk terlalu sedikit atau terlalu bnyk.

Stlh pengujian, nilai min_support 0.1 dipilih krn menghasilkan daftar kelompok produk yg dianggap ideal. Hasil output menunjukkan bahwa 'Selai' menempati urutan teratas dgn nilai support 0.52, yg berarti muncul dlm 52% dari total data. Selain itu, terlihat pula pola kombinasi sprt 'Selai' dn 'Teh' yg memiliki frekuensi kemunculan cukup signifikan bagi analisis asosiasi selanjutnya.

# Langkah 4: Bentuk & Saring Aturan Asosiasi
Bentuk aturan asosiasi, saring dengan min_confidence dan min_lift, lalu urutkan
berdasarkan Lift tertingg

In [20]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
'support', 'confidence', 'lift']].head(10))
# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

         antecedents consequents  support  confidence      lift
10       (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
15  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
11      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
13     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
9      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
14   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


Catatan : Berdasarkan hasil analisis, ditemukan bbrp pola belanja yg menarik. Aturan paling kuat ditunjukkan oleh kombinasi (Teh, Keju) -> (Telur) dgn nilai Lift tertinggi mencapai 2.38. Hal ini menandakan bahwa pelanggan yg membeli Teh dn Keju memiliki kemungkinan 2.38 kali lipat lbh besar utk jg membeli Telur dibandingkan pelanggan biasa.

pola (Roti) -> (Selai) sangat masuk akal krn kedua produk tsb saling melengkapi (komplementer). Dgn nilai Confidence 0.68, dpt disimpulkan bahwa 68% orang yg membeli Roti jg akan mengambil Selai.

# Langkah 5: Rekomender Sederhana dengan Content-Based Filtering
Bangun katalog produk dengan kategori, lalu buat rekomendasi produk serupa
menggunakan cosine similarity atas kategori (one-hot).

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
'produk': produk,
'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
  idx = katalog.index[katalog['produk'] == nama_produk][0]
  skor = list(enumerate(sim_matrix[idx]))
  skor = sorted(skor, key=lambda x: x[1], reverse=True)
  skor = [s for s in skor if s[0] != idx][:top_n]
  return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


Catatan : Cell kode ini menerapkan sistem rekomendasi sederhana berbasis fitur produk atau Content-Based Filtering. Pertama, katalog produk disusun dgn mengelompokkan setiap barang ke dlm kategori tertentu sprt Bakery, Dairy, atau Minuman. Selanjutnya, kategori tsb diubah menjadi bentuk angka (one-hot encoding) agar dpt dihitung kemiripannya menggunakan metode Cosine Similarity.

sistem akan mencari produk lain yg memiliki kategori sama dgn produk yg dipilih. Sebagai contoh, saat mencari produk yg mirip dgn 'Roti', output menampilkan 'Selai', 'Sereal', dn 'Susu'. Hal ini terjadi krn produk-produk tsb berada dlm kelompok fitur yg serupa, sehingga dpt direkomendasikan sebagai alternatif atau pelengkap belanja.

# Langkah 6: Bandingkan Kedua Pendekatan
Bandingkan rekomendasi dari aturan asosiasi (Langkah 4) dengan rekomendasi ContentBased (Langkah 5) untuk produk yang sama

In [22]:
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
  lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))
# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Catatan : Cell kode terakhir ini membandingkan hasil rekomendasi dari dua metode berbeda utk produk 'Roti'. Terlihat bahwa kedua pendekatan memberikan hasil yg cukup konsisten krn sama-sama merekomendasikan 'Selai'. ini membuktikan bahwa pola belanja aktual dlm transaksi (Association Rules) sejalan dgn kemiripan kategori produk (Content-Based).

Penggunaan Association Rules sngt efektif jika data transaksi sdh melimpah krn mampu menangkap perilaku nyata pembeli. Sementara itu, Content-Based lbh cocok digunakan saat menghadapi produk baru yg blm memiliki riwayat transaksi. Utk hasil terbaik, penggabungan keduanya (hybrid) sngt disarankan agar sistem tdk hny mengandalkan kategori barang saja, tp jg mengikuti tren kebiasaan konsumen di lapangan.

# **Kesimpulan**
Di hands on ini kita telah mengintegrasikan analisis asosiasi dn sistem rekomendasi dlm satu alur kerja. Melalui algoritma Apriori, ditemukan pola belanja signifikan sprt kombinasi (Roti) -> (Selai) yg diperkuat dgn nilai Lift tinggi, menunjukkan hubungan erat antar produk dlm transaksi nyata.

Di sisi lain, implementasi Content-Based Filtering memberikan alternatif rekomendasi berdasarkan kemiripan kategori produk, yg terbukti konsisten dgn pola transaksi yg ada. Kesimpulannya, penggabungan kedua metode menciptakan sistem rekomendasi yg tangguh: mampu menangkap perilaku konsumen sekaligus tetap efektif dlm menyarankan produk baru meskipun riwayat transaksinya msh minim.